# Accuracy diagnostics by item popularity

This notebook supplements (and does not replace or alter) the primary held-out accuracy metrics. It decomposes P@10 and Recall@10 into Head/Mid/Tail item contributions, reports the share of liked test items in each group, and computes clipped inverse-popularity-weighted Recall@10 plus Tail-only NDCG@10.

The analysis uses the same split, item popularity labels, and saved recommendation lists from one calibration run. The settings analyzed are read from the run's `selected_on_validation.csv` (and cross-checked against the manifest's `selectedSettings`), so the notebook covers exactly the settings the calibration notebook selected on validation and exported. Change the run ID in the configuration cell to analyze another compatible run.

**Interpretation limits:** test relevance is observational and popularity-skewed. Inverse-propensity weighting is only an offline proxy: item propensity is proportional to its training rating count, and normalized inverse weights are clipped to `[1, 10]` (most-popular item weight 1). The clip limits the influence of rare items but does not correct exposure or rating-selection bias. These diagnostics do not measure perceived calibration, user satisfaction, or causal preference.

**Observed seed-42 conclusion (run 2026-11-16T01-57-11, validation-selected settings):** 26.2% of liked test pairs are Head, 58.8% Mid, and 15.0% Tail, while 64.1% of baseline recommendation slots are Tail. All three selected CP/HA settings (`cp_l0.9`, `ha_n100_l0.9`, `ha_n50_l0.9`) improve clipped inverse-propensity-weighted Recall@10 over baseline, but all three reduce Tail-only NDCG@10. Thus their aggregate accuracy gain persists under this specific debiasing proxy, while the conclusion does not extend to Tail relevance and remains sensitive to the test set's popularity mix. The settings compared are those selected on validation, so a different run may report different findings; read them from the printed comparison in section 4. This is not evidence of perceived calibration.


In [ ]:
import $ivy.`org.apache.spark::spark-sql:3.5.9`

import org.apache.spark.sql.{DataFrame, SparkSession}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.types._

val spark = SparkSession.builder()
  .appName("CS 7265 - Accuracy Popularity Diagnostics")
  .master("local[2]")
  .config("spark.driver.host", "127.0.0.1")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
import spark.implicits._
require(spark.version == "3.5.9", s"Expected Spark 3.5.9, found ${spark.version}")
println(s"PASS: Spark ${spark.version}, Scala ${scala.util.Properties.versionNumberString}")

## 1. Configuration and run provenance

The settings compared are the ones the calibration run selected on validation, read from `selected_on_validation.csv` and cross-checked against the manifest. The notebook verifies the run's manifest seed and recommendation inputs before computing diagnostics.

In [ ]:
val calibrationRunId = "2026-11-16T01-57-11.331215Z"
val expectedSplitSeed = 42L
val listSize = 10
val ratingThreshold = 4.0
val inverseWeightClip = 10.0

val ratingsPath = Seq("data/ml-1m/ratings.dat", "../data/ml-1m/ratings.dat")
  .find(path => new java.io.File(path).isFile)
  .getOrElse(throw new IllegalArgumentException("Run from the repository root or notebooks folder with data/ml-1m/ present."))
val repositoryRoot = new java.io.File(ratingsPath).getCanonicalFile
  .getParentFile.getParentFile.getParentFile
val calibrationDir = new java.io.File(repositoryRoot, s"outputs/calibration/$calibrationRunId")
require(calibrationDir.isDirectory, s"Calibration run not found: ${calibrationDir.getPath}")
val manifestPath = new java.io.File(calibrationDir, "manifest.txt").toPath
require(java.nio.file.Files.isRegularFile(manifestPath), s"Missing manifest: $manifestPath")
// Normalize Windows line endings so the key=value checks below also work for a checked-out manifest.
val manifest = new String(java.nio.file.Files.readAllBytes(manifestPath), "UTF-8").replace("\r\n", "\n")
require(manifest.contains(s"splitSeed=$expectedSplitSeed\n"),
  s"Calibration run does not use splitSeed=$expectedSplitSeed")
val inputsDir = new java.io.File(calibrationDir, "inputs")
val manifestValues = manifest.split("\n").flatMap { line =>
  val separator = line.indexOf('=')
  if (separator > 0) Some(line.substring(0, separator) -> line.substring(separator + 1)) else None
}.toMap
val profileRunId = manifestValues.getOrElse("profileRunId",
  throw new IllegalArgumentException("Calibration manifest has no profileRunId."))
val profileDir = new java.io.File(repositoryRoot, s"outputs/popularity-profiles/$profileRunId")
require(profileDir.isDirectory, s"Popularity profile run not found: ${profileDir.getPath}")
val movieGroups = spark.read.parquet(new java.io.File(profileDir, "movie-popularity-groups").getPath)
  .select(col("movieId").cast(IntegerType), col("ratingCount").cast(LongType), col("popularityGroup"))
val evaluationUsers = spark.read.parquet(new java.io.File(inputsDir, "evaluation-users").getPath)
  .select(col("userId").cast(IntegerType)).distinct()
// The settings to analyze are the validation-selected ones the calibration run exported.
val selectedCsvFile = new java.io.File(calibrationDir, "selected_on_validation.csv")
require(selectedCsvFile.isFile, s"Missing selected-settings export: ${selectedCsvFile.getPath}")
val selectedLines = new String(java.nio.file.Files.readAllBytes(selectedCsvFile.toPath), "UTF-8")
  .split("\r?\n").toList
require(selectedLines.nonEmpty, s"${selectedCsvFile.getName} is empty.")
val selectedHeader = selectedLines.head.split(",", -1).toSeq
def selectedColumn(name: String): Int = {
  val index = selectedHeader.indexOf(name)
  require(index >= 0, s"${selectedCsvFile.getName} has no '$name' column.")
  index
}
val settingColumn = selectedColumn("setting")
val methodColumn = selectedColumn("method")
val lambdaColumn = selectedColumn("lambda")
val n0Column = selectedColumn("n0")
val settings = selectedLines.tail.filter(_.nonEmpty).map { line =>
  val fields = line.split(",", -1)
  (fields(settingColumn), fields(methodColumn), fields(lambdaColumn).toDouble, fields(n0Column).toInt)
}.distinct.sortBy { case (setting, method, _, _) => (if (method == "baseline") 0 else 1, setting) }
require(settings.nonEmpty, s"${selectedCsvFile.getName} lists no settings.")
require(settings.map(_._1).distinct.size == settings.size,
  s"${selectedCsvFile.getName} lists the same setting name with different parameters.")
require(settings.exists(_._1 == "baseline"), "The selected settings must include the baseline.")
val manifestSelectedSettings = manifestValues.getOrElse("selectedSettings",
  throw new IllegalArgumentException("Calibration manifest has no selectedSettings.")).split(",").map(_.trim).toSet
require(manifestSelectedSettings == settings.map(_._1).toSet,
  s"Manifest selectedSettings $manifestSelectedSettings differ from ${selectedCsvFile.getName}: ${settings.map(_._1).toSet}")
settings.foreach { case (setting, _, _, _) =>
  require(new java.io.File(calibrationDir, s"top-10-$setting").isDirectory,
    s"Missing recommendation export for $setting in ${calibrationDir.getPath}")
}
println(s"PASS: run $calibrationRunId, seed $expectedSplitSeed, ${settings.size} selected settings: ${settings.map(_._1).mkString(", ")}")

## 2. Rebuild the test relevance set

Recreate the canonical per-user split from the local ratings file using the seed and training fraction recorded in the calibration manifest. Relevant items are ratings `>= 4.0`, matching the primary evaluation. Popularity groups and rating counts come from the run's training-only movie table.

In [ ]:
val trainingFraction = manifestValues.getOrElse("trainingFraction",
  throw new IllegalArgumentException("Calibration manifest has no trainingFraction.")).toDouble
val ratingSchema = StructType(Seq(
  StructField("userId", IntegerType, false), StructField("movieId", IntegerType, false),
  StructField("rating", FloatType, false), StructField("timestamp", LongType, false)
))
val ratings = spark.read.option("sep", "::").schema(ratingSchema).csv(ratingsPath)
val perUser = Window.partitionBy("userId")
val ranked = ratings
  .withColumn("userRatingCount", count(lit(1)).over(perUser))
  .withColumn("shuffleKey", xxhash64(lit(expectedSplitSeed), col("userId"), col("movieId")))
  .withColumn("ratingPosition", row_number().over(
    Window.partitionBy("userId").orderBy(col("shuffleKey"), col("movieId"))))
  .withColumn("trainingCount", floor(col("userRatingCount") * trainingFraction))
val trainingRatings = ranked.filter(col("ratingPosition") <= col("trainingCount"))
val testRatings = ranked.filter(col("ratingPosition") > col("trainingCount"))
val trainingCount = trainingRatings.count()
val testCount = testRatings.count()
assert(trainingCount == manifestValues("trainingRatings").toLong)
assert(testCount == manifestValues("testRatings").toLong)
val likedAll = testRatings.filter(col("rating") >= ratingThreshold).select(col("userId"), col("movieId"))
val maxTrainingRatingCount = movieGroups.agg(max(col("ratingCount"))).head().getLong(0)
require(maxTrainingRatingCount > 0L, "Training popularity counts must be positive.")
// Keep every liked test pair, including movies the training split never saw. Those have no popularity
// label or rating count, so they get group "U" (unseen) and the largest clipped weight. They can never be
// recommended, so they never score a hit, but they stay in each user's recall denominator.
val likedWithWeights = likedAll.join(movieGroups, Seq("movieId"), "left")
  .withColumn("ratingCount", coalesce(col("ratingCount"), lit(0L)))
  .withColumn("popularityGroup", coalesce(col("popularityGroup"), lit("U")))
  .withColumn("inversePropensityWeight",
    when(col("ratingCount") <= 0L, lit(inverseWeightClip))
      .otherwise(least(lit(inverseWeightClip), lit(maxTrainingRatingCount.toDouble) / col("ratingCount"))))
  .select(col("userId"), col("movieId"), col("popularityGroup"), col("ratingCount"), col("inversePropensityWeight"))
val relevantCount = likedWithWeights.count()
val unseenLikedCount = likedWithWeights.filter(col("popularityGroup") === "U").count()
assert(relevantCount == likedAll.count(), "The popularity join must not add or drop liked test pairs.")
assert(relevantCount > 0L)
assert(likedWithWeights.filter(col("popularityGroup") =!= "U" && col("ratingCount") <= 0L).count() == 0L)
println(s"PASS: split counts train=$trainingCount, test=$testCount; liked test pairs=$relevantCount (of which $unseenLikedCount are not in the training split)")

## 3. Popularity-decomposed and debiased metrics

For H/M/T, P@10 contribution is that group's hits divided by 10, averaged over users with at least one liked test item. Recall@10 contribution is that group's hits divided by each such user's total liked test items, averaged over those users; the three contributions sum to ordinary Recall@10. Liked test movies that the training split never saw have no popularity label, so they are labeled `U` (unseen): they stay in every user's denominator (so the group contributions still sum to the primary Recall@10), can never be hit, count toward the liked-test base rate (so the H/M/T shares sum to 1 minus the unseen share), and get the largest clipped weight in the weighted recall. Tail-only NDCG@10 uses only items labeled Tail. The liked-test share is the fraction of all liked user–item test pairs in each group; recommendation share is the fraction of all recommendation slots in that group.

For the supplementary inverse-propensity-weighted recall, each liked item receives `min(10, maxTrainingRatingCount / itemTrainingRatingCount)`. Each user's weighted hits are divided by the total clipped weights of their liked test items, then averaged over users with at least one liked test item. Tail-only NDCG@10 treats only liked Tail items as relevant and averages over users with at least one liked Tail test item. These metrics are diagnostics, not replacements for the primary metrics.

In [ ]:
val recommendations = settings.map { case (setting, _, _, _) =>
  spark.read.parquet(new java.io.File(calibrationDir, s"top-10-$setting").getPath)
    .select(col("userId").cast(IntegerType), col("movieId").cast(IntegerType),
      col("recommendationRank").cast(IntegerType))
    .withColumn("setting", lit(setting))
}.reduce(_.unionByName(_))
val settingNames = settings.map(_._1).toDF("setting")
val actualUsers = recommendations.select("userId").distinct()
assert(actualUsers.count() == evaluationUsers.count(), "Recommendation user count differs from evaluator cohort.")
assert(actualUsers.except(evaluationUsers).count() == 0L && evaluationUsers.except(actualUsers).count() == 0L,
  "Recommendation users differ from the expected evaluation cohort.")
val invalidLists = recommendations.groupBy("setting", "userId")
  .agg(count(lit(1)).as("rows"), countDistinct(col("movieId")).as("movies"),
    countDistinct(col("recommendationRank")).as("ranks"))
  .filter(col("rows") =!= listSize || col("movies") =!= listSize || col("ranks") =!= listSize)
assert(invalidLists.count() == 0L, "Every recommendation list must contain 10 unique items and ranks.")
val hitRows = recommendations.join(likedWithWeights, Seq("userId", "movieId"), "inner")
val recommendationCounts = recommendations.join(movieGroups.select("movieId", "popularityGroup"), Seq("movieId"))
  .groupBy("setting", "popularityGroup").agg(count(lit(1)).as("recommendedItems"))
  .withColumn("recommendationShare", col("recommendedItems") /
    lit(listSize.toDouble * evaluationUsers.count().toDouble))
assert(recommendationCounts.groupBy("setting").agg(sum(col("recommendationShare")).as("shareTotal"))
  .filter(abs(col("shareTotal") - 1.0) > 1e-12).count() == 0L,
  "Recommendation popularity shares must sum to 1 for every setting.")
val hitCounts = hitRows.groupBy("setting", "userId", "popularityGroup")
  .agg(count(lit(1)).as("hits"))
val userRelevant = likedWithWeights.groupBy("userId")
  .agg(count(lit(1)).as("relevantItems"), sum(col("inversePropensityWeight")).as("weightedRelevantItems"))
val settingEligibleUsers = settingNames.crossJoin(userRelevant)
val groupNames = Seq("H", "M", "T").toDF("popularityGroup")
val groupGrid = settingEligibleUsers.crossJoin(groupNames)
  .join(hitCounts, Seq("setting", "userId", "popularityGroup"), "left")
  .withColumn("hits", coalesce(col("hits"), lit(0L)))
val groupContributions = groupGrid.groupBy("setting", "popularityGroup")
  .agg(avg(col("hits") / lit(listSize.toDouble)).as("precisionContribution"),
    avg(col("hits") / col("relevantItems")).as("recallContribution"),
    countDistinct(col("userId")).as("scoredUsers"))
val baseRates = likedWithWeights.groupBy("popularityGroup")
  .agg(count(lit(1)).as("likedItems"))
  .withColumn("likedTestShare", col("likedItems") / lit(relevantCount.toDouble))
val popularityShareTotal = baseRates.agg(sum(col("likedTestShare"))).head().getDouble(0)
assert(math.abs(popularityShareTotal - 1.0) < 1e-12,
  s"H/M/T liked-test base rates sum to $popularityShareTotal, not 1.")
val groupResults = groupContributions.join(baseRates, Seq("popularityGroup"))
  .join(recommendationCounts, Seq("setting", "popularityGroup"))
  .select("setting", "popularityGroup", "likedTestShare", "recommendationShare",
    "precisionContribution", "recallContribution", "scoredUsers")
val weightedHits = hitRows.groupBy("setting", "userId")
  .agg(sum(col("inversePropensityWeight")).as("weightedHits"))
val ipwRecall = settingEligibleUsers.join(weightedHits, Seq("setting", "userId"), "left")
  .withColumn("weightedHits", coalesce(col("weightedHits"), lit(0.0)))
  .withColumn("ipwRecall", col("weightedHits") / col("weightedRelevantItems"))
  .groupBy("setting").agg(avg(col("ipwRecall")).as("ipwRecall"),
    countDistinct(col("userId")).as("ipwScoredUsers"))
val tailLikedItems = likedWithWeights.filter(col("popularityGroup") === "T")
  .select("userId", "movieId")
val tailRelevant = tailLikedItems.groupBy("userId")
  .agg(count(lit(1)).as("tailRelevantItems"))
  .withColumn("idcg", expr("aggregate(sequence(1, least(tailRelevantItems, 10)), cast(0.0 as double), (acc, x) -> acc + 1.0 / log2(x + 1))"))
val tailHits = recommendations.join(tailLikedItems, Seq("userId", "movieId"), "inner")
  .groupBy("setting", "userId")
  .agg(sum(lit(1.0) / log2(col("recommendationRank") + 1.0)).as("dcg"))
val tailNdcg = settingNames.crossJoin(tailRelevant)
  .join(tailHits, Seq("setting", "userId"), "left")
  .withColumn("dcg", coalesce(col("dcg"), lit(0.0)))
  .groupBy("setting").agg(avg(col("dcg") / col("idcg")).as("tailNdcg"),
    countDistinct(col("userId")).as("tailScoredUsers"))
val overallContributions = groupResults.groupBy("setting")
  .agg(sum(col("precisionContribution")).as("precisionContribution"),
    sum(col("recallContribution")).as("recallContribution"))
val overallResults = overallContributions.join(ipwRecall, Seq("setting"))
  .join(tailNdcg, Seq("setting"))
  .join(settings.map { case (setting, method, lambda, n0) =>
    (setting, method, lambda, n0)
  }.toDF("setting", "method", "lambda", "n0"), Seq("setting"))
// The decomposition must reproduce the primary test metrics exported by the calibration run.
val primaryMetrics: Map[String, (Double, Double, Int)] = selectedLines.tail.filter(_.nonEmpty).map(_.split(",", -1))
  .filter(f => f(selectedColumn("groupType")) == "all")
  .map(f => f(settingColumn) -> (f(selectedColumn("testPrecision")).toDouble,
    f(selectedColumn("testRecall")).toDouble, f(selectedColumn("testScoredUsers")).toInt)).toMap
overallResults.select("setting", "precisionContribution", "recallContribution", "ipwScoredUsers").collect().foreach { row =>
  val setting = row.getAs[String]("setting")
  val (primaryPrecision, primaryRecall, primaryScoredUsers) = primaryMetrics.getOrElse(setting,
    throw new IllegalStateException(s"No primary test metrics for $setting in ${selectedCsvFile.getName}."))
  assert(math.abs(row.getAs[Double]("precisionContribution") - primaryPrecision) < 1e-9,
    s"$setting: P@10 contributions do not sum to the primary P@10 $primaryPrecision.")
  assert(math.abs(row.getAs[Double]("recallContribution") - primaryRecall) < 1e-9,
    s"$setting: Recall@10 contributions do not sum to the primary Recall@10 $primaryRecall.")
  assert(row.getAs[Long]("ipwScoredUsers") == primaryScoredUsers.toLong,
    s"$setting: scored users differ from the primary evaluation ($primaryScoredUsers).")
}
println("PASS: P@10 and Recall@10 decompositions and scored-user counts match the primary test metrics")
println("PASS: all recommendation lists have 10 unique items and match the expected user cohort")
groupResults.orderBy("setting", "popularityGroup").show(100, truncate = false)
overallResults.orderBy("setting").show(20, truncate = false)

## 4. Compare gains with baseline and save supplementary CSV

The comparison table reports CP/HA minus baseline changes in inverse-propensity-weighted Recall@10 and Tail-only NDCG@10. Positive values mean an increase in that metric. Treat the results as robust to this particular offline proxy only when the gains remain positive; improvements in ordinary P@10/Recall@10 alone may reflect the test set's popularity mix. No claim here establishes how users perceive calibration.

In [ ]:
val overallRows = overallResults.collect().map { row =>
  (row.getAs[String]("setting"), row.getAs[String]("method"),
    row.getAs[Double]("lambda"), row.getAs[Int]("n0"),
    row.getAs[Double]("precisionContribution"), row.getAs[Double]("recallContribution"),
    row.getAs[Double]("ipwRecall"), row.getAs[Double]("tailNdcg"),
    row.getAs[Long]("ipwScoredUsers"), row.getAs[Long]("tailScoredUsers"))
}
val baselineMetrics = overallRows.find(_._1 == "baseline")
  .getOrElse(throw new IllegalStateException("Baseline metrics are missing."))
val comparisonRows = overallRows.filter(_._1 != "baseline").map { row =>
  (row._1, row._2, row._3, row._4, row._7 - baselineMetrics._7, row._8 - baselineMetrics._8,
    row._5 - baselineMetrics._5, row._6 - baselineMetrics._6)
}
val comparisonBySetting = comparisonRows.map(row => row._1 -> row).toMap
println("Gain over baseline: positive debiased recall / Tail NDCG gains persist under these proxies.")
comparisonRows.foreach { row =>
  val ipwVerdict = if (row._5 > 0.0) "PERSISTS" else "does not persist"
  val tailVerdict = if (row._6 > 0.0) "PERSISTS" else "does not persist"
  println(f"${row._1}%-16s delta IPW Recall=${row._5}%+.6f ($ipwVerdict), delta Tail NDCG=${row._6}%+.6f ($tailVerdict), delta P@10=${row._7}%+.6f, delta Recall=${row._8}%+.6f")
}
val outputPath = new java.io.File(calibrationDir, "accuracy-popularity-diagnostics.csv").toPath
val groupRows = groupResults.collect().map { row =>
  val setting = row.getAs[String]("setting")
  val config = settings.find(_._1 == setting).get
  Seq(setting, config._2, config._3.toString, config._4.toString,
    row.getAs[String]("popularityGroup"), row.getAs[Double]("likedTestShare").toString,
    row.getAs[Double]("recommendationShare").toString,
    row.getAs[Double]("precisionContribution").toString,
    row.getAs[Double]("recallContribution").toString, "", "",
    row.getAs[Long]("scoredUsers").toString, "", "", "", "", "").mkString(",")
}
val overallCsvRows = overallRows.map { row =>
  val deltas = comparisonBySetting.get(row._1)
  val deltaIpw = deltas.map(_._5.toString).getOrElse("")
  val ipwPersists = deltas.map(value => (value._5 > 0.0).toString).getOrElse("")
  val deltaTail = deltas.map(_._6.toString).getOrElse("")
  val tailPersists = deltas.map(value => (value._6 > 0.0).toString).getOrElse("")
  Seq(row._1, row._2, row._3.toString, row._4.toString, "all", "", "", row._5.toString,
    row._6.toString, row._7.toString, row._8.toString, row._9.toString,
    row._10.toString, deltaIpw, ipwPersists, deltaTail, tailPersists).mkString(",")
}
val csvHeader = "setting,method,lambda,n0,group,likedTestShare,recommendationShare,precisionContribution,recallContribution,ipwRecall,tailNdcg,scoredUsers,tailScoredUsers,deltaIpwRecallVsBaseline,ipwRecallGainPersists,deltaTailNdcgVsBaseline,tailNdcgGainPersists"
val csvRows = groupRows ++ overallCsvRows
val csvText = (Seq(csvHeader) ++ csvRows).mkString("\n") + "\n"
java.nio.file.Files.write(outputPath, csvText.getBytes("UTF-8"),
  java.nio.file.StandardOpenOption.CREATE_NEW, java.nio.file.StandardOpenOption.WRITE)
require(java.nio.file.Files.size(outputPath) > 0L, "Supplementary diagnostics CSV was not written.")
println(s"PASS: saved supplementary diagnostics to $outputPath")
spark.stop()